# LFW Grad-CAM — 06. Representative case visualization

모든 집단 분석이 끝난 뒤에만 stable/high-error/rank-flip/
threshold-crossing 예시를 선택합니다. 이 단계는 이미 저장된 heatmap을
읽어 그림만 만들며 Grad-CAM을 다시 생성하지 않습니다.


In [1]:
# cell 1 : 환경 설정 및 profile 검증
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_PROFILE = "arcface_ms1mv3_r100"     # arcface, adaface, magface 중 이번 실행 profile
MODE = "dev"               # 빠른 검증은 dev, 전체 논문 실행만 real
DATA_FRACTION = 1       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합과 random control의 재현 seed
EXECUTE_STAGE = True      # 필수 입력을 채우고 이 단계 계산 시에만 True
WRITE_OUTPUTS = True      # 새 immutable artifact 저장 시에만 True

all_profiles = CONFIG["models"]["selected_profiles"] + CONFIG["models"].get("bridge_profiles", [])
available_profiles = CONFIG["models"]["profiles"]
blocked_profiles = CONFIG["models"].get("blocked_profiles", [])

if MODEL_PROFILE in blocked_profiles:
    raise RuntimeError(f"차단된 profile입니다: {MODEL_PROFILE}")
if MODEL_PROFILE not in available_profiles:
    raise ValueError(f"지원하지 않는 MODEL_PROFILE: {MODEL_PROFILE}")
PROFILE = available_profiles[MODEL_PROFILE]
MODEL_FAMILY = PROFILE["family"]
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")


In [2]:
# cell 2 : 입력 경로 및 import
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from research.explainability.gradcam import (
    read_population_heatmaps,
    select_population_representative_cases,
)

JOINED_METRICS_PATH = None
SALIENCY_ARTIFACT_DIR = None
SELECTED_MANIFEST_PATH = None
ALIGNED_FACES_NPY_PATH = None
CASE_MANIFEST_OUTPUT_PATH = None
FIGURE_OUTPUT_DIR = None
CASES_PER_GROUP = int(
    CONFIG["gradcam"]["representative_case_visualization"][
        "samples_per_stratum"
    ]
)

# 자동 기본값 및 경로 탐색
if ALIGNED_FACES_NPY_PATH is None:
    ALIGNED_FACES_NPY_PATH = PROJECT_ROOT / CONFIG["datasets"]["lfw"]["aligned_crops"]["array_path"]

if JOINED_METRICS_PATH is None or SALIENCY_ARTIFACT_DIR is None or SELECTED_MANIFEST_PATH is None:
    lfw_runs_root = PROJECT_ROOT / CONFIG["run"]["root"] / "lfw"
    latest_joined = sorted(lfw_runs_root.rglob("joined_saliency_compression.parquet"), key=lambda p: p.stat().st_mtime, reverse=True)
    if latest_joined:
        run_dir = latest_joined[0].parent
        if JOINED_METRICS_PATH is None:
            JOINED_METRICS_PATH = latest_joined[0]
        if SALIENCY_ARTIFACT_DIR is None:
            SALIENCY_ARTIFACT_DIR = run_dir / "saliency_population"
        if SELECTED_MANIFEST_PATH is None:
            SELECTED_MANIFEST_PATH = run_dir / "selected_manifest.parquet"
        if CASE_MANIFEST_OUTPUT_PATH is None:
            CASE_MANIFEST_OUTPUT_PATH = run_dir / "representative_cases.parquet"
        if FIGURE_OUTPUT_DIR is None:
            FIGURE_OUTPUT_DIR = run_dir / "representative_figures"


In [3]:
# cell 3 : 대표 사례 선택 및 시각화
if EXECUTE_STAGE:
    required = {
        "joined": JOINED_METRICS_PATH,
        "saliency": SALIENCY_ARTIFACT_DIR,
        "selected": SELECTED_MANIFEST_PATH,
        "aligned_faces": ALIGNED_FACES_NPY_PATH,
    }
    missing = [name for name, value in required.items() if value is None]
    if missing:
        raise RuntimeError(f"입력 경로가 비어 있습니다: {missing}")
    joined = pd.read_parquet(required["joined"])
    cases = select_population_representative_cases(
        joined,
        cases_per_group=CASES_PER_GROUP,
        seed=SEED,
    )
    heatmap_ids, heatmaps = read_population_heatmaps(
        required["saliency"]
    )
    heatmap_index = {
        str(sample_id): index
        for index, sample_id in enumerate(heatmap_ids.astype(str))
    }
    selected = pd.read_parquet(required["selected"])
    selected["sample_id"] = selected["sample_id"].astype(str)
    aligned = np.load(
        required["aligned_faces"],
        mmap_mode="r",
        allow_pickle=False,
    )
    case_summary = {
        "case_count": int(len(cases)),
        "case_groups": cases["case_group"].value_counts().to_dict(),
        "regenerated_gradcam": False,
    }
    if WRITE_OUTPUTS:
        if CASE_MANIFEST_OUTPUT_PATH is None or FIGURE_OUTPUT_DIR is None:
            raise RuntimeError("case manifest와 figure 출력 경로를 지정하세요.")
        case_path = Path(CASE_MANIFEST_OUTPUT_PATH).resolve()
        figure_dir = Path(FIGURE_OUTPUT_DIR).resolve()
        if case_path.exists() or figure_dir.exists():
            raise FileExistsError("기존 case 결과를 덮어쓸 수 없습니다.")
        case_path.parent.mkdir(parents=True, exist_ok=True)
        figure_dir.mkdir(parents=True)
        cases.to_parquet(case_path, index=False)
        sample_to_face = selected.set_index("sample_id")[
            "aligned_face_index"
        ].astype(int).to_dict()
        for row in cases.itertuples(index=False):
            sample_id = str(row.sample_id)
            image = np.asarray(aligned[sample_to_face[sample_id]])
            heatmap = heatmaps[heatmap_index[sample_id]]
            figure, axis = plt.subplots(figsize=(3, 3))
            axis.imshow(image)
            axis.imshow(
                heatmap,
                cmap="jet",
                alpha=0.45,
                extent=(0, image.shape[1], image.shape[0], 0),
            )
            axis.set_title(f"{row.case_group}: {sample_id}")
            axis.axis("off")
            figure.savefig(
                figure_dir / f"{row.case_id}.png",
                dpi=160,
                bbox_inches="tight",
            )
            plt.close(figure)
else:
    cases = pd.DataFrame()
    case_summary = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
display(cases.head(20))
case_summary


,case_id,sample_id,compression_family,compression_profile,origin_dimension,search_dimension,reconstruction_available,metric_vector_source,reconstruction_mse,angular_error_rad,...,high_saliency_occlusion_score_drop,low_saliency_occlusion_score_drop,random_occlusion_score_drop,random_occlusion_score_drop_std,faithfulness_gain_over_low_saliency,faithfulness_gain_over_random,case_group,case_priority_rank,selection_seed,cases_per_group
0,gradcam-38756abf93f0454b2ee8,lfw:Bill_Simon:Bill_Simon_0001,origin,origin_512,512,512,True,reconstruction,0.000000,0.000537,...,0.059297,0.011794,0.016850,0.014685,0.047503,0.042447,high_error,1,42,8
1,gradcam-4c5876c23c4d851093dc,lfw:Ishaq_Shahryar:Ishaq_Shahryar_0001,origin,origin_512,512,512,True,reconstruction,0.000000,0.000526,...,0.051354,0.025813,0.032435,0.010963,0.025541,0.018919,high_error,2,42,8
2,gradcam-92950995182b058f867a,lfw:Jane_Pauley:Jane_Pauley_0002,origin,origin_512,512,512,True,reconstruction,0.000000,0.000514,...,0.053003,0.077794,0.107143,0.019998,-0.024791,-0.054139,high_error,3,42,8
3,gradcam-bec07ca06a3b3a3a9ccd,lfw:Anneli_Jaatteenmaki:Anneli_Jaatteenmaki_0002,origin,origin_512,512,512,True,reconstruction,0.000000,0.000511,...,0.084410,0.017201,0.074474,0.029432,0.067209,0.009936,high_error,4,42,8
4,gradcam-fc35213b16f8d84401da,lfw:Antonio_Palocci:Antonio_Palocci_0005,origin,origin_512,512,512,True,reconstruction,0.000000,0.000505,...,0.013593,0.031189,0.035814,0.011417,-0.017597,-0.022221,high_error,5,42,8
5,gradcam-e1f5b7605d8dae52a08f,lfw:Monica_Seles:Monica_Seles_0006,origin,origin_512,512,512,True,reconstruction,0.000000,0.000498,...,0.055983,0.060521,0.041077,0.014060,-0.004539,0.014905,high_error,6,42,8
6,gradcam-a04b3a21f7087ba746dd,lfw:Diana_Krall:Diana_Krall_0004,origin,origin_512,512,512,True,reconstruction,0.000000,0.000496,...,0.177301,0.089208,0.095798,0.017710,0.088093,0.081503,high_error,7,42,8
7,gradcam-70fd927bae8a3c1a114b,lfw:Jude_Law:Jude_Law_0001,origin,origin_512,512,512,True,reconstruction,0.000000,0.000489,...,0.037226,0.033646,0.067735,0.017955,0.003579,-0.030509,high_error,8,42,8
8,gradcam-a6d55239dd240a548c42,lfw:Salman_Rushdie:Salman_Rushdie_0002,origin,origin_512,512,512,True,reconstruction,0.000000,0.000000,...,0.093652,0.028690,0.012131,0.008105,0.064962,0.081521,stable,1,42,8
9,gradcam-d96225bba9116c2a0d30,lfw:Condoleezza_Rice:Condoleezza_Rice_0010,origin,origin_512,512,512,True,reconstruction,0.000000,0.000000,...,0.174036,0.024255,0.027710,0.019431,0.149781,0.146327,stable,2,42,8


{'case_count': 96,
 'case_groups': {'high_error': 48, 'stable': 48},
 'regenerated_gradcam': False}

사례 그림은 집단 통계의 보조 설명입니다. 사례에서 보인 패턴을 전체
이미지의 일반적 원인으로 확대 해석하지 않습니다.
